# 📖 **Input/Output & Attributes**

> ### 📌 **TL;DR**
>
> This Jupyter notebook has been created to compare different features with several open source Python libraries for rasters management.
>
> This notebook focuses on raster reading/writing and metadata handling (CRS, transform, nodata, dtype, main attributes) across several libraries.</br>
> The goal is to highlight differences in data models, ergonomics, and metadata propagation during I/O operations.
>
> The following libraries will be considered :
>- `rasterio`
>- `rioxarray`
>- `odc-geo`
>- `geoutils`

In [ ]:
import rasterio
from rasterio.plot import show

import xarray as xr

import rioxarray

import odc.geo.xr

import geoutils as gu

import matplotlib.pyplot as plt
import matplotlib.patches as patches

import seaborn as sns

import dask

In [ ]:
#path to the raster object
raster_path = "../data/rasters/105005005BDEA700-visual.tif"

## **Opening the raster object and checking its main attributes**

### • *rasterio*

The dedicated raster object for `rasterio` is a `rasterio.Dataset`.

We can open our dataset by using the [`rasterio.open()`](https://rasterio.readthedocs.io/en/stable/api/rasterio.html#rasterio.open) function. 

The returned object is a `DatasetReader`, which allows to have access to its metadata and already plot the image if necessary. In addition, with `.open()`, the image is still not loaded into memory (lazy behaviour).

In [ ]:
ds_rasterio = rasterio.open(raster_path)

In [ ]:
print(type(ds_rasterio))

We can directly plot the image to get an overview of it, by using [`show()`](https://rasterio.readthedocs.io/en/stable/api/rasterio.plot.html#rasterio.plot.show) :

In [ ]:
show(ds_rasterio.read(1), cmap="mako")

Main attributes can be accessed individually :

In [ ]:
print(f"""
RASTERIO DATASET ATTRIBUTES\n
Path : {ds_rasterio.name}
Driver : {ds_rasterio.driver}
Number of band(s) : {ds_rasterio.count}
Data type : {ds_rasterio.dtypes}
Size : {ds_rasterio.height} rows x {ds_rasterio.width} columns
CRS : {ds_rasterio.crs}
Bounds : {ds_rasterio.bounds}
Transform : {ds_rasterio.transform}
Nodata : {ds_rasterio.nodata}
""")

We can also use `.meta.items()` to iterate through each item of the metadata dictionary and improve the output :

In [ ]:
print("RASTERIO DATASET ATTRIBUTES\n")
for key, value in ds_rasterio.meta.items():
    print(f"{key} : {value}")

To access the actual data, we have to use the [`.read()`](https://rasterio.readthedocs.io/en/stable/api/rasterio.io.html#rasterio.io.BufferedDatasetWriter.read) method on our DatasetReader previously created. The `.read()` method, unlike `.open`, will load the rasters data into memory.

We can directly specify (or not) the band we want to access, as follows :

In [ ]:
data = ds_rasterio.read()

band1 = ds_rasterio.read(1)  # first band

In [ ]:
band1

The object created is a `numpy.ndarray` containing the value of each pixel of our dataset.

### • *rioxarray*

The dedicated raster object for rioxarray is a `xarray.DataArray` enriched with an accessor `.rio`.

We can use the [`open_rasterio()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.open_rasterio) function to create our `DataArray` object and access its different attributes. As for `rasterio`, `open_rasterio()` will not load the data into memory until requested (lazy behaviour).

One of the advantage of using rioxarray is the possibility to divide our data by `chunks`. Chunks are like small blocks of data that will allow us to work with rasters bigger than our RAM, but also to do parallel computing (giving tasks to different processors at the same time). Instead of loading a full raster into memory, `dask` will only processes the needed chunks.

To create our chunked array, we need to specify the chunks sizes along each dimension we want, using the `chunks` argument. The size of each chunk along dimension can be given specifically, e.g. `{'x' : 5, 'y' : 5}`, but can also be set to `True` or `"auto"` to be selected automatically. In the example below we will keep it to `True` to select sizes automatically.

In [ ]:
da_rioxarray_true = rioxarray.open_rasterio(raster_path, chunks=True)

In [ ]:
da_rioxarray_true

Trying to set `chunks` to `"auto"` seems to provide the same result as `True`:

In [ ]:
da_rioxarray_auto = rioxarray.open_rasterio(raster_path, chunks="auto")

In [ ]:
da_rioxarray_auto

As we can see, the created object is a `xarray.DataArray`, which is a multidimensional array (usually `bands` x `height` x `width`). It contains, among others, the coordinates (actual X/Y values) and the attributes which are metadata of the raster.

Here, we see that our chunks have the size of 121 MB (`11264 x 11264`), which might be too big for a single image (but not the case for big collections) and can lead to memory overload. This comes from the fact that the original raster is not chunked on disk. [Choosing the good size is not trivial](https://blog.dask.org/2021/11/02/choosing-dask-chunk-sizes), but we consider `2048 x 2048` to be generally a good compromise for most dataset, so we will create a new array with those parameters :

In [ ]:
da_rioxarray = rioxarray.open_rasterio(raster_path, chunks={'x' : 2048, 'y' : 2048})

In [ ]:
da_rioxarray

Now, we see that our chunks are much better distributed along our dataset and have the size of `12MB`, which is more optimized for the memory and parallel computing (the ideal recommanded size is generally between `10~200 MB` depending on the size of the dataset).
However, rechunking is a heavy task that must be avoided if possible and that can take a bit of time, so choosing the good chunk size at the beginning of the computation is essential!

We can now access main attributes individually, as follows :

In [ ]:
print(f"""
RIOXARRAY DATASET ATTRIBUTES\n
Dimensions : {da_rioxarray.dims}
Number of band(s) : {da_rioxarray.rio.count}
Data type : {da_rioxarray.dtype}
Shape : {da_rioxarray.shape}
Size : {da_rioxarray.rio.height} rows x {da_rioxarray.rio.width} columns
CRS : {da_rioxarray.rio.crs}
Resolution : {da_rioxarray.rio.resolution()}
Bounds : {da_rioxarray.rio.bounds()}
Transform : {da_rioxarray.rio.transform()}
Nodata : {da_rioxarray.rio.nodata}
Attributes : {da_rioxarray.attrs}
""")

As in `rasterio`, we can see the main attributes by using `.meta.items()` to iterate over the metadata dictionary.

To access the data, we can use different methods :

In [ ]:
da_rioxarray.data

`.data` will return a dask.array.Array if the input dataset is chunked (and has not been computed). This method is useful to manipulate the data and perform calculations without overload memory.

In [ ]:
da_rioxarray.values

`.values` is an eager method that will always return a `numpy.ndarray` by loading all the data into memory, even if the dataset is chunked.

In [ ]:
da_rioxarray.compute()

`.compute()` will return a `xarray.DataArray` and will compute the graph stored in the array and therefore load all the data into memory.

To access data of a specific pixel, we can use `.isel().item()` (selecting by index/ position) :

In [ ]:
da_rioxarray.isel(band=0, y=100, x=200).compute().item()

We can also access one specific band by using `.sel()` (selection by real coordinates) and specify the band we want : 

In [ ]:
da_rioxarray.sel(band=1)

### • *odc-geo*

To open a raster, `odc-geo` uses function from other packages. Here we can use the rioxarray `.open_rasterio()` function :

In [ ]:
ds_odcgeo = rioxarray.open_rasterio(raster_path)

In [ ]:
ds_odcgeo

Some attributes can be obtained using the odc `.odc` accessor, as follows :

In [ ]:
print(f"""ODC-GEO DATASET ATTRIBUTES\n
CRS : {ds_odcgeo.odc.crs}
Transform : {ds_odcgeo.odc.transform}
Geobox : {ds_odcgeo.odc.geobox}
Nodata : {ds_odcgeo.odc.nodata}
""")

We can see that `odc.geo` concepts differ a bit from the ones developped by `rasterio` and `rioxarray` (and its underlying `GDAL` layer): the transform georeferencing model is completed by a geobox.

In [ ]:
ds_odcgeo.data

### • *geoutils*

The dedicated raster object for geoutils is a `geoutils.Raster`.

In [ ]:
ds_gu = gu.Raster(raster_path)

In [ ]:
ds_gu

A detailed overview of the main attributes is obtained using the `.info()` function :

In [ ]:
print("GEOUTILS DATASET ATTRIBUTES\n")
print(ds_gu.info())

Raster image can easily be plotted with [`.plot()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.plot.html#geoutils.Raster.plot):

In [ ]:
ds_gu.plot(bands=1, cmap="mako")

We can access the data of our raster with `.data` :

In [ ]:
ds_gu.data

It returns two elements : an array with the actual values for each pixel, and an array with the boolean matrix of the raster (True if the pixel has no data).

## **Access to GCPs (Ground Control Points)**

Access to GCPs are implemented in `rasterio`, `rioxarray` and `geoutils`, but not in `odc-geo`.

Ground Control Points are points that connect a pixel to real coordinates (geographic or projected).

### • *rasterio*

GCPs in rasterio can be accessed with [`.get_gcps()`](https://rasterio.readthedocs.io/en/stable/api/rasterio.io.html#rasterio.io.BufferedDatasetWriter.get_gcps).

It will return a tuple in the following format : (`gcp_list`, `gcp_crs`).

In [ ]:
with rasterio.open("../data/rasters/IMAGERY.TIF") as src:
    gcps, crs_gcps = src.get_gcps()

In [ ]:
print(gcps)
print(crs_gcps)

Each GCP is an object `rasterio.control.GroundControlPoint` containing :
- *row, col*
- *x, y, z*
- *id, info*

### • *rioxarray*

In [ ]:
ds_rioxarray = rioxarray.open_rasterio("../data/rasters/IMAGERY.TIF")

With rioxarray, we can have access to GCPs by using the rio accessor, with [`.get_gcps()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.rioxarray.XRasterBase.get_gcps) as follows :

In [ ]:
ds_rioxarray.rio.get_gcps()

It returns a list of all GCPs with their coordinates.

## **Access to RPCs (Rational Polynomial Coefficients)**

Access to RPCs are only implemented in rasterio.

RPCs can also be used to georeference a raster.

### • *rasterio*

In rasterio, they can be checked using [`.rpcs`](https://rasterio.readthedocs.io/en/stable/api/rasterio.io.html#rasterio.io.BufferedDatasetWriter.rpcs).

In [ ]:
with rasterio.open("../data/rasters/IMG_SPOT7_MS_201602150257025_SEN_1671661101_R1C1.JP2") as src:
    rpcs = src.rpcs

In [ ]:
print(rpcs)

In [ ]:
print(type(rpcs))

It will return an instance of `rasterio.rpc.RPC` if there are RPCs, or empty otherwise.

## **Save to local disk**

In this section, we will demonstrate how to save our raster locally.

### • *rasterio*

First, we need to get all metadata of our raster with the profile.

Then, to save a raster locally with rasterio, we will use `.open()` to open our raster as `'w'` (writing mode).

We can then save the output file with the `.write()` function :

In [ ]:
with rasterio.open(raster_path) as src:
    data = src.read()
    profile = src.profile

    profile.update(
        dtype=rasterio.uint8,
        count=3,
        compress="JPEG")

    with rasterio.open('../outputs/rasterio_output.tif', 'w', **profile) as dst:
        dst.write(data)

### • *rioxarray*

With rioxarray, we can save our raster locally with the function [`to_raster()`](https://corteva.github.io/rioxarray/html/rioxarray.html#rioxarray.raster_array.RasterArray.to_raster).

Unlike rasterio, no need to save metadata first :

In [ ]:
with rioxarray.open_rasterio(raster_path) as src:
    src.rio.to_raster("../outputs/rioxarray_raster_output.tif", compress="JPEG")

### • *geoutils*

Using geoutils, we can save our raster locally by using the [`save()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.save.html#geoutils.Raster.save) function:

In [ ]:
ds_gu.save("../outputs/geoutils_raster.tif")

## **Save to COG (Cloud-Optimized GeoTIFF) format**

A Cloud-Optimized GeoTIFF (COG) is a regular GeoTIFF file optimized for remote access, structured to be read efficiently from the cloud, without the need to download the entire file.

### • *rasterio*

Since recent versions of GDAL (> 3.1), rasterio can use the driver `COG` when saving a file.

In [ ]:
from rasterio.shutil import copy as rio_copy

input_raster_path = raster_path
output_rasterio_cog = "../outputs/rasterio_cog.tif"

# COG options
cog_profile = dict(
    driver="COG",
    compress="LZW",
    blocksize=512,
    overview_resampling="average"
)

rio_copy(
    input_raster_path,
    output_rasterio_cog,
    **cog_profile
)

We can verify the validity of our COG file newly created using the `cog_validate()` function from the `rio-cogeo` library.

In [ ]:
from rio_cogeo.cogeo import cog_validate

cog_validate(output_rasterio_cog, strict=True)

### • *rioxarray*

It is possible to save COG files using rioxarray as well with the function [`to_raster()`](https://rasterio.readthedocs.io/en/stable/api/rasterio.io.html#rasterio.io.BufferedDatasetWriter.rpcs) since it integrates the driver `COG` as for rasterio :

In [ ]:
output_rioxarray_cog = "../outputs/rioxarray_cog.tif"

with rioxarray.open_rasterio(raster_path) as src:
    src.rio.to_raster(
        output_rioxarray_cog,
        driver="COG",
        compress="LZW",
        blocksize=512,
        BIGTIFF="IF_SAFER",
        overview_resampling="average"
    )

Once again, we can verify the validity of the COG file with `rio-cogeo` :

In [ ]:
cog_validate(output_rioxarray_cog, strict=True)

### • *odc-geo*

`odc-geo` includes several functions to save COG files, with or without `dask`.

- Without dask, using the [`write_cog()`](https://odc-geo.readthedocs.io/en/latest/_api/odc.geo.xr.write_cog.html) function:

In [ ]:
from odc.geo.xr import write_cog

In [ ]:
da_odcgeo = rioxarray.open_rasterio(raster_path)

output_odcgeo_cog = "../outputs/odcgeo_cog.tif"

write_cog(da_odcgeo, output_odcgeo_cog)

We can check the validity using :

In [ ]:
cog_validate(output_odcgeo_cog, strict=True)

### • *geoutils*

With geoutils, we will be using the `.save` function, but this time we specify `driver="COG` as an argument :

In [ ]:
ds_gu.save("../outputs/geoutils_cog.tif", driver="COG")

In [ ]:
cog_validate("../outputs/geoutils_cog.tif", strict=True)